# Colab run: ResNet-50 vs ViT-B/16 (focal loss)
Runtime -> Change runtime type -> **GPU** first. Run the cells in order. Everything is saved to Google Drive so a disconnect does not lose results.

You need: (1) your `kaggle.json`, (2) `fitzpatrick17k_mappable.zip` (154 MB, made locally) uploaded to `MyDrive/skinfair/` on Drive.

In [ ]:
import torch
assert torch.cuda.is_available(), 'Turn on a GPU: Runtime > Change runtime type'
print(torch.cuda.get_device_name(0))

## 1. Drive + repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/skinfair'
import os; os.makedirs(DRIVE, exist_ok=True)

In [ ]:
!git clone https://github.com/tanamikaa/skin-lesion-fairness.git
%cd skin-lesion-fairness
!pip install -q -r requirements.txt && pip install -q -e .

## 2. Data
HAM10000 from Kaggle (upload `kaggle.json` when asked). Fitzpatrick17k from the zip on Drive.

In [ ]:
from google.colab import files
files.upload()  # choose kaggle.json
!mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!pip install -q kaggle
!kaggle datasets download -d kmader/skin-cancer-mnist-ham10000 -p data/HAM10000 --unzip
import shutil, pathlib
dst = pathlib.Path('data/HAM10000/images'); dst.mkdir(exist_ok=True)
for f in pathlib.Path('data/HAM10000').rglob('*.jpg'):
    if f.parent != dst: shutil.move(str(f), dst / f.name)
print(len(list(dst.glob('*.jpg'))))  # must be 10015

In [ ]:
!unzip -q -o {DRIVE}/fitzpatrick17k_mappable.zip -d data/
!ls data/fitzpatrick17k/images | wc -l   # must be 1916
!ls data/HAM10000/HAM10000_metadata.csv

In [ ]:
import logging; logging.basicConfig(level=logging.INFO)
from skinfair.data.datasets import Fitzpatrick17kDataset, load_ham_metadata
print(len(load_ham_metadata('data/HAM10000')), len(Fitzpatrick17kDataset('data/fitzpatrick17k')))  # 10015, 1916

## 3. Quick smoke test (1 epoch, ~2 min on a T4)

In [ ]:
!python scripts/train.py --config configs/resnet50.yaml --set train.epochs=1 data.num_workers=2 output_dir=/content/smoke checkpoint_dir=/content/smoke/ckpt

## 4. Real runs
Same budget for both models. `EPOCHS` applies to both; change it only here. Outputs go to Drive: `results/<model>/<run_name>/` and `ckpt/<run_name>_best.pt`.

In [ ]:
EPOCHS = 15
COMMON = f'train.epochs={EPOCHS} data.num_workers=2 checkpoint_dir={DRIVE}/ckpt'
!python scripts/train.py --config configs/resnet50.yaml --set {COMMON} output_dir={DRIVE}/results/resnet50

In [ ]:
!python scripts/train.py --config configs/vit_b16.yaml --set {COMMON} output_dir={DRIVE}/results/vit_b16

## 5. Evaluate (predictions + metrics.json) and compare

In [ ]:
R = f'{DRIVE}/results'
!python scripts/evaluate.py --config configs/resnet50.yaml --checkpoint {DRIVE}/ckpt/resnet50_focal_s42_best.pt --set output_dir={R}/resnet50 data.num_workers=2
!python scripts/evaluate.py --config configs/vit_b16.yaml --checkpoint {DRIVE}/ckpt/vit_b16_focal_s42_best.pt --set output_dir={R}/vit_b16 data.num_workers=2

In [ ]:
!python scripts/compare_models.py --a {R}/resnet50/resnet50_focal_s42/predictions_fitz.csv --b {R}/vit_b16/vit_b16_focal_s42/predictions_fitz.csv --name-a ResNet-50 --name-b ViT-B/16 --out {DRIVE}/figures

## 6. Explanations (Grad-CAM / attention rollout) + deletion/insertion quality

In [ ]:
!python scripts/explain.py --config configs/resnet50.yaml --checkpoint {DRIVE}/ckpt/resnet50_focal_s42_best.pt --split fitz --max-images 256 --set output_dir={R}/resnet50 data.num_workers=2
!python scripts/explain.py --config configs/vit_b16.yaml --checkpoint {DRIVE}/ckpt/vit_b16_focal_s42_best.pt --split fitz --max-images 256 --set output_dir={R}/vit_b16 data.num_workers=2
!cp reports/figures/overlay_*.png {DRIVE}/figures/

## 7. Look at the results, then download the folder `MyDrive/skinfair` to your laptop
Copy `metrics.json`, `log.csv` (best val balanced accuracy / epoch) and the figures into the repo's `results/` and `reports/figures/`, then fill `results/EXPERIMENTS.md`.

In [ ]:
import json
for m, r in [('resnet50', 'resnet50_focal_s42'), ('vit_b16', 'vit_b16_focal_s42')]:
    d = json.load(open(f'{R}/{m}/{r}/metrics.json'))
    print(m, 'HAM test bal-acc', round(d['ham_test']['balanced_accuracy'], 3),
          '| light-dark gap', d['fitzpatrick17k'].get('light_minus_dark_gap_ci'))